# Module 1: LLM Basics — Talking to the Brain

# What is an LLM?

An **LLM (Large Language Model)** is a model trained on huge amounts of text that can read a prompt and generate a relevant reply. In an AI Agent, the LLM is the **"brain"** — the part that actually reasons, decides, and writes things. Every other piece we build in this notebook (tools, state, loops) exists purely to support this brain: feeding it information, letting it take actions, and remembering what happened.

# Why Are We Using Groq?

For this whole project, we're using **Groq** as our LLM provider, with the **llama-3.1-8b-instant** model specifically.

- Groq doesn't train its own models — it runs already-existing open models (like Meta's Llama models) on special hardware called LPUs (Language Processing Units), which makes responses extremely fast.
- "llama-3.1-8b-instant" = Llama version 3.1, 8 Billion parameters (a small, fast model), "instant" meaning it's tuned for low response time.
- Speed matters a LOT for an agent — a single user query might need several separate calls to the LLM (one call per reasoning step, as we'll see later in this notebook with the ReAct pattern). A slow LLM = a slow agent.
- Groq has a generous free tier, which is great while we're learning.

# How Do We Talk to an LLM From Code?

Groq follows the same style as OpenAI's API: you send a **list of messages**, where every message is a dictionary with:
- `"role"` → who is "speaking" (`"system"`, `"user"`, or `"assistant"`)
- `"content"` → what they said

This "messages" format is the industry standard for chat-based LLMs — you'll see this exact same shape again and again in every agent framework out there (LangChain, LangGraph, CrewAI, OpenAI's SDK, etc.), so it's worth getting comfortable with it right from this very first cell.

# Before We Code: Get a Groq API Key

1. Go to https://console.groq.com
2. Create a free account
3. Go to "API Keys" and create a new key
4. Add it to your `.env` file as: `GROQ_API_KEY=your_key_here`

We never want to hardcode an API key directly in a notebook — if you ever share this file, anyone with your key could use up your free quota (or worse).

In [ ]:
import os #Read environment variables

from dotenv import load_dotenv #Load .env file

from groq import Groq # to access the Groq API (this gives us llama-3.1-8b-instant, a fast open-source LLM)

print("Success")

load_dotenv() #loads the variables from our .env file into the environment, so os.getenv() can read them below

groq_api_key = os.getenv("GROQ_API_KEY") # groq api key, read from the environment instead of typed directly here

client = Groq(api_key=groq_api_key) #Connects the Groq SDK with our API key - this "client" is our open phone line to Groq's servers

response = client.chat.completions.create(
    model="llama-3.1-8b-instant", #the LLM we are using - Meta's Llama 3.1 8B model, hosted by Groq for very fast responses (LLM = Brain)
    messages=[
        {"role": "user", "content": "What is an AI Agent?"} #sending our query to the LLM, in the "messages" format Groq expects
    ]
)

print(response.choices[0].message.content)

# What Just Happened?

We just made our very first call to an LLM:

1. `os` let us read environment variables (like our API key) without typing them directly in the notebook.
2. `load_dotenv()` loaded the contents of our `.env` file into those environment variables.
3. We created a `client` — our open connection to Groq's servers.
4. We sent one message (`role: "user"`) asking "What is an AI Agent?"
5. Groq ran that message through `llama-3.1-8b-instant` and sent back a reply, which we reached via `response.choices[0].message.content`.

This is the simplest possible interaction with an LLM — no tools, no memory, no looping yet. Just "ask a question, get a text answer." Everything else in this notebook is about giving this same basic call more structure: letting the model choose actions (Tool Selection), remembering what happened (State), and looping until it's done (ReAct + Agent Loop).

In [95]:
import math


class CalculatorTool: #A class is a blueprint. It defines a set of attributes and methods that the objects created from the class will have.

    def run(self, expression): #This is the function the agent will call.

        try:
            return eval(expression, {"math": math}) #expression means the mathematical expression that the agent will evaluate. 
            #eval() is a built-in function in Python that parses the expression passed to it and executes it as a Python expression. 
            # #It can be used to evaluate simple mathematical expressions, but it should be used with caution as it can execute arbitrary code if the input is not controlled.
            #math is a module in Python that provides mathematical functions and constants. By passing {"math": math} as the second argument to eval(),
            #  we are providing access to the math module's functions and constants within the evaluated expression. which will be used to calculate square roots, trigonometric functions, etc.
        except Exception:
            return "Calculation Error"

In [96]:
calc = CalculatorTool()

print(calc.run("250*80"))

20000


In [97]:
#to use the Search tool, we will use the DuckDuckGoSearchRun class from the langchain_community.tools module. 
#This class provides a simple interface for performing web searches using the DuckDuckGo search engine.
from langchain_community.tools import DuckDuckGoSearchRun

In [98]:
# real search tool class that the agent will use to perform web searches.
class SearchTool:

    def __init__(self): #initialization method for the SearchTool class. This method is called when an instance of the class is created. 
        #It initializes the search attribute with an instance of the DuckDuckGoSearchRun class, which will be used to perform web searches.

        self.search = DuckDuckGoSearchRun()

    def run(self, query):

        return self.search.run(query) #this line calls the run method of the DuckDuckGoSearchRun instance (self.search) with the provided query.

In [99]:
search = SearchTool()

result = search.run("CEO of Google")

print(result)

7 Apr 2026 ... Sundar Pichai is the CEO of Google and Alphabet. He sits down with John and Elad Gil to discuss Google's resurgence in the AI race, managing a massive $180 ... 19 May 2026 ... Editor's note: Below is an edited transcript of Google CEO Sundar Pichai's remarks at Google I/O 2026, adapted to include more of what was announced on ... 21 hours ago ... Google CEO and Stanford alumnus Sundar Pichai had been invited to deliver the commencement address to the graduating Class of 2026. As he began speaking, ... 17 hours ago ... BBC questions Google CEO Sundar Pichai after grads walk out during his speech | BBC News India ... This content isn't available. Skip video. 30 Apr 2026 ... When Pichai took over as CEO in 2015 from co-founder Larry Page, he had a vision in mind. Most people still treated thinking machines as sci-fi fantasy. So it ...


In [100]:
import math

def population_sqrt_workflow(city):#we have created a workflow function that the agent can call to get the population of a city. 
    #The agent will use the Search tool to get the population information.

    search = SearchTool()
    calc = CalculatorTool()

    population = search.run(f"{city} population")

    print("Population:", population)

    square_root = calc.run(f"math.sqrt({population})")

    print("Square Root:", square_root)

    return square_root

In [101]:
population_sqrt_workflow("Chennai")

Population: Chennai is home to a diverse population of ethno-religious communities, with a majority Hindu population (80.7%). The city houses nearly three million migrant population and has the third-largest expatriate population in India. Chennai metropolitan area or Greater Chennai is the fourth-most populous metropolitan area in India and the 35th most populous in the world. It consists of the core city of Chennai, which is coterminous with the Chennai district, and its suburbs in Kanchipuram, Chengalpattu, Thiruvallur and Ranipet districts. Chennai has an estimated population of 4.9 million, with an area that has grown from 176 square kilometers to 426 square kilometers after a 2011 expansion. The urban agglomeration, which includes the city and suburbs, has a population estimated at 9 million. Chennai is a major metropolitan area in India, situated in the Tamil Nadu area. With a population of 6,599,000 people, it represents approximately 8.7% of the total urban population in India

'Calculation Error'

In [103]:
#prompt for the agent to select a tool based on the user's query
tool_selection_prompt = """
You are an AI Agent.

Available Tools:

1. Calculator
- Use for mathematical calculations.

2. Search
- Use for factual questions and information retrieval.

Return the answer in JSON format without fences.

Examples:

User Query: What is 250 * 80?

{
    "reason": "This is a mathematical calculation.",
    "tool": "Calculator",
    "input": "250*80"
}

User Query: Who is the CEO of Google?

{
    "reason": "This is a factual question.",
    "tool": "Search",
    "input": "CEO of Google"
}
"""

In [104]:
#query for the agent to process
query = "What is 250 * 80?"

In [ ]:
#Agent selects a tool based on the query and the prompt
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": tool_selection_prompt + f"\n\nUser Query: {query}"}
        #combining our instructions (tool_selection_prompt) with the user's actual question into ONE message,
        #since we haven't introduced separate system/user messages yet at this point in the notebook
    ]
) #sending the combined prompt and user query to the LLM for processing

response_text = response.choices[0].message.content #pulling out just the text reply, same idea as our LLM Basics cell above

In [ ]:
print(response_text)

In [107]:
import json

In [ ]:
tool_data = json.loads(response_text)

print(tool_data)

In [109]:
selected_tool = tool_data["tool"]
tool_input = tool_data["input"]

print(selected_tool)
print(tool_input)

Calculator
250*80


In [110]:
if selected_tool == "Calculator": #if the selected tool is Calculator, then create an instance of the CalculatorTool class 

    calc = CalculatorTool() #and call the run method with the tool_input as the argument

    result = calc.run(tool_input)

    print(result)

20000


# What Have We Built Now?

Architecture:

User Query

      ↓

Groq (Llama 3.1)

      ↓

Choose Tool

      ↓

Provide Tool Input

      ↓

Python Executes Tool

      ↓

Result

This is the first time your agent is actually taking an action, not just reasoning.

# Next Step: Create run_agent()

Right now you're running multiple cells manually:

query = ...

esponse = ...

tool_data = ...

selected_tool = ...

result = ...

A real agent should hide all of that inside one function.

so we are going to combine all these functions as one below 

In [111]:
# we have a state dictionary to keep track of the agent's interactions and decisions  added with reasoning and observation.
# This state can be used for debugging, analysis, or even for learning purposes in more advanced
state = {
    "query": "",
    "reason": "",
    "selected_tool": "",
    "tool_input": "",
    "observation": "",
    "result": "",
    "steps": []
}

In [ ]:
def run_agent(query):

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": tool_selection_prompt + f"\n\nUser Query: {query}"}
        ]
    )

    tool_data = json.loads(response.choices[0].message.content) #parse the JSON response from the LLM to extract the selected tool and its input

    #state updates with reasoning 
    state["steps"] = [] #we have added this to avoid appending to the steps list from previous runs of the agent. 
    #This ensures that each run of the agent starts with a clean slate, and the steps list only contains the steps taken during the current run.

    state["query"] = query
    state["reason"] = tool_data["reason"]
    state["selected_tool"] = tool_data["tool"]
    state["tool_input"] = tool_data["input"]


    selected_tool = state["selected_tool"]
    tool_input = state["tool_input"]

    if selected_tool == "Calculator": #if the selected tool is Calculator, then create an instance of the CalculatorTool class

        calc = CalculatorTool()

        result = calc.run(tool_input)

        state["observation"] = result #update the state with the observation (result) obtained from the tool
        state["result"] = result

        state["steps"].append({
            "reason": state["reason"],
            "tool": state["selected_tool"],
            "input": state["tool_input"],
            "observation": state["observation"]
        })
        return result
    
    elif selected_tool == "Search": #for web search queries, if the selected tool is Search, then create an instance of the SearchTool class

        search = SearchTool()

        result = search.run(tool_input)

        state["observation"] = result #update the state with the observation (result) obtained from the tool
        state["result"] = result

        state["steps"].append({
            "reason": state["reason"],
            "tool": state["selected_tool"],
            "input": state["tool_input"],
            "observation": state["observation"]
        })
        return result

In [113]:
# Run the agent with a sample query for calculator
answer = run_agent(
    "What is 250 * 80?"
)

print(answer)

20000


In [114]:
# run the agent with another sample query for web search
run_agent("Who is the CEO of Google?")

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 51.263100701s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 51
}
]

# What is State?

State is simply the agent's working memory for the current task.

# Without state:

User Query

↓

Tool Result

↓

Forgot everything

# With state:

User Query

↓

Store in State

↓

Tool Result

↓

Store in State

↓

Next Step can use it

In [ ]:
#we have a state dictionary to keep track of the agent's interactions and decisions. 
# This state can be used for debugging, analysis, or even for learning purposes in more advanced implementations. 
# The state dictionary is updated with the user's query, the selected tool, the input to the tool, and the result obtained from the tool.
state["query"]
state["selected_tool"]
state["tool_input"]
state["result"]

'Search Result: CEO of Google'

In [ ]:
print(state)
print(json.dumps(state, indent=4)) #it is a built-in function in Python that converts a Python object into a JSON string. 
#The indent parameter is used to specify the number of spaces to use for indentation in the resulting JSON string, making it more readable.

{'query': 'Who is the CEO of Google?', 'reason': 'This is a factual question.', 'selected_tool': 'Search', 'tool_input': 'CEO of Google', 'observation': 'Search Result: CEO of Google', 'result': 'Search Result: CEO of Google', 'steps': [{'reason': 'This is a factual question.', 'tool': 'Search', 'input': 'CEO of Google', 'observation': 'Search Result: CEO of Google'}]}
{
    "query": "Who is the CEO of Google?",
    "reason": "This is a factual question.",
    "selected_tool": "Search",
    "tool_input": "CEO of Google",
    "observation": "Search Result: CEO of Google",
    "result": "Search Result: CEO of Google",
    "steps": [
        {
            "reason": "This is a factual question.",
            "tool": "Search",
            "input": "CEO of Google",
            "observation": "Search Result: CEO of Google"
        }
    ]
}


 till now we have completed the state, next we have to insert REACTLOOP method into the agent

 Why ReAct Matters

Imagine a harder task:

Find Chennai's population and calculate its square root.

The agent could do:

Reason:
Need population first.

Act:
Search Tool

Observe:
11,933,000

Reason:
Now calculate square root.

Act:
Calculator Tool

Observe:
3454.4

Final Answer:
3454.4

we altered the early created state with reasonoing and observation and the added them in agent tool function

we have to create multi step react for the collbaorative work between different tools

# Multi-Step ReAct

The agent should do:

Reason:
Need Chennai population first.

Act:
Search("Chennai population")

Observe:
11,933,000

Reason:
Now calculate square root.

Act:
Calculator("sqrt(11933000)")

Observe:
3454.4

Final Answer:
3454.4

we have added the steps to state

In [ ]:
run_agent("Who is the CEO of Google?")
print(json.dumps(state, indent=4))

{
    "query": "Who is the CEO of Google?",
    "reason": "This is a factual question.",
    "selected_tool": "Search",
    "tool_input": "CEO of Google",
    "observation": "Search Result: CEO of Google",
    "result": "Search Result: CEO of Google",
    "steps": [
        {
            "reason": "This is a factual question.",
            "tool": "Search",
            "input": "CEO of Google",
            "observation": "Search Result: CEO of Google"
        }
    ]
}


currently our agent will use only one tool at a time for one query, if user give "find chennai population and get square root of it' it can run another tool by observing the previous tools result 

In [ ]:
#creating a workflow function that the agent can call to get the population of a city. The agent will use the Search tool to get the population information.
def population_sqrt_workflow(city):

    search = SearchTool()

    population = search.run(f"{city} population")

    print("Population:", population)

    return population

In [ ]:
population_sqrt_workflow("Chennai")

Population: Search Result: Chennai population


'Search Result: Chennai population'

# The ReAct Pattern — Letting the Agent Think in Loops

# The Problem With Our Current Agent

Look back at our `run_agent()` function from earlier. It does exactly ONE thing:

User Query
   ↓
Pick ONE tool
   ↓
Run that ONE tool
   ↓
Return the result

That's perfectly fine for "What is 250 * 80?" — one calculation, one tool, done.

But what about: **"Find Chennai's population and calculate its square root"?**

That actually needs TWO steps:
1. **Search tool** → find the population
2. **Calculator tool** → calculate the square root of that population

Our current agent has no way to do step 2, because it forgets everything the moment step 1 finishes. This is exactly the wall we hit earlier with our hand-written `population_sqrt_workflow()` function — we had to hardcode the steps ourselves in Python, instead of letting the agent figure them out on its own. A hardcoded workflow only works for the ONE specific task we wrote it for. A real agent should be able to handle a task we never explicitly coded for.

# What is ReAct?

**ReAct** stands for **Re**ason + **Act**. It's a prompting pattern (from a well-known research paper) where, instead of asking the LLM for one final answer straight away, we ask it to work in a repeating loop:

Thought → "What should I do next, based on everything I know so far?"
   ↓
Action → "Use this tool, with this input"
   ↓
Observation → "Here's what the tool returned"
   ↓
(back to Thought, now using the new Observation)
   ↓
... repeat as many times as needed ...
   ↓
Final Answer

# Why This Matters for Real AI Agents

This loop is the actual engine behind almost every "AI Agent" you've heard of — AutoGPT, LangChain agents, OpenAI's function-calling agents, customer-support bots that look things up before replying, coding agents that run a command and react to the output. None of them magically know the full answer in one shot. They reason, act, observe, and repeat — exactly what we're about to build.

# How We'll Implement It

We'll keep a running text log called a **scratchpad**. Every Thought / Action / Observation gets appended to it, and we feed the WHOLE scratchpad back to the LLM on every loop iteration. This matters because LLMs have no memory of their own between separate API calls — the only "memory" the agent has is whatever text we choose to feed back in. The scratchpad IS the agent's memory of the current task.

In [ ]:
# this is the system prompt that teaches the LLM HOW to behave as a ReAct agent.
# unlike our earlier tool_selection_prompt (which only ever picked ONE tool and stopped), this prompt also
# teaches the model about a 3rd "tool" called Final Answer, which lets the model decide for ITSELF when it
# already has enough information to stop looping and answer the user.

react_prompt = """
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

1. Calculator
- Use for mathematical calculations.

2. Search
- Use for factual questions and information retrieval.

3. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{
    "reason": "why you are taking this step",
    "tool": "Calculator" or "Search",
    "input": "the input to give the tool"
}

If you already know the final answer:
{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a tool call you already made.
- Give the Final Answer as soon as you have enough information. Don't waste extra steps.
"""

# What Just Happened?

We wrote a new system prompt, `react_prompt`, that is more powerful than our earlier `tool_selection_prompt`:

- it still describes the **Calculator** and **Search** tools the same way as before
- but it adds a 3rd option, **Final Answer**, which is not a "real" tool at all — it's how the model tells us "I'm done, here's the answer"
- it explicitly tells the model that it will be shown the conversation **so far** (the scratchpad), and must look at it before deciding the next step

This single prompt is what makes the multi-step loop possible — the LLM itself decides, step by step, whether to keep gathering information or to stop and answer.

In [ ]:
# small helper function so we don't repeat the same 3 lines of Groq code every time we want to talk to the LLM.
# this is good practice: if we ever want to change the model name, or add a setting like temperature, we only
# have to change it in ONE place instead of hunting through the whole notebook.

def call_llm(messages):  # messages = the list of {"role": ..., "content": ...} dictionaries we want to send

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages
    )

    return response.choices[0].message.content  # we only need the text of the reply, not the whole response object

# What Just Happened?

We wrapped the Groq API call we wrote earlier into a reusable function, `call_llm()`. It takes a list of messages (the same `messages` format we used in our very first Groq test) and returns just the plain text reply.

From here on, instead of writing `client.chat.completions.create(...)` everywhere, we'll just write `call_llm(messages)`. This is the same idea as our `CalculatorTool` and `SearchTool` classes earlier — wrap something we'll reuse a lot into one clean, named piece of code.

# The Agent Loop

Now we put everything together. An **Agent Loop** is just a normal Python loop (a `for` or `while`) that keeps going around and around:

1. Send the current scratchpad to the LLM
2. Ask "what's your next step?"
3. Run whatever tool the LLM picked
4. Add the result (the Observation) to the scratchpad
5. Repeat — UNTIL the LLM says **Final Answer**, or we hit a safety limit

This loop is the actual "agent" part of an AI Agent. Everything before this (tools, state, prompts) was just preparing the ingredients - the loop is what actually makes decisions repeatedly and adapts based on what it learns.

# Why a Safety Limit (`max_steps`)?

Without a limit, a bug (or a confused model) could loop forever — calling the API again and again, costing time and money, and never actually answering the user. `max_steps` works like a circuit breaker: if the agent hasn't produced a Final Answer within, say, 5 steps, we stop it ourselves and admit we couldn't get a clean answer, instead of looping endlessly.

# Where Final Answer Generation Fits In

Notice that we don't need a *separate* "Final Answer step" bolted on afterwards — Final Answer Generation is simply ONE of the branches inside this same loop. The model produces it naturally, the moment it decides it has enough information. This is exactly how Module 9 (Final Answer Generation) and Module 8 (Agent Loop) are built — together, as one piece.

# A State Management Gotcha: Stale Keys

Before we run the agent again, there's a bug worth understanding (you may have already spotted it if you printed `state` after switching from `run_agent()` to `run_react_agent()`).

`run_react_agent()` only ever writes three keys into `state`: `"query"`, `"steps"`, and `"result"`. But our earlier `run_agent()` function wrote OTHER keys into the very same `state` dictionary — `"reason"`, `"selected_tool"`, `"tool_input"`, `"observation"`. Since both functions share one global `state` dict, and `run_react_agent()` never touches those older keys, they just sit there.

The result: if you call `run_agent("Who is the CEO of Google?")` and then later call `run_react_agent("Find Chennai's population and calculate the square root of it.")`, printing `state` afterward shows a confusing mix — `"query"` correctly says Chennai, but `"tool_input"` / `"observation"` / `"result"` are still leftover values about the Google CEO question. It looks like the tool input doesn't match the query, but really it's just two different functions' outputs sitting in the same dictionary at once.

# The Fix

We already follow the "clean slate every run" principle for `state["steps"]` inside `run_agent()`. We just extend that same idea to the WHOLE dictionary using `state.clear()` at the very start of `run_react_agent()` — that way, whatever we print afterward only ever reflects the run we just did, with nothing left over from any previous call (to either agent function).

In [ ]:
def run_react_agent(query, max_steps=5):
    # max_steps is our safety limit (the "circuit breaker" explained earlier) - default 5 is plenty for the
    # kind of 1-2 tool tasks we're testing with.

    # wipe out the ENTIRE state dictionary before this run starts, not just "steps". This matters because
    # run_agent() (and even a previous run_react_agent() call) can leave behind keys like "reason",
    # "selected_tool", "tool_input", "observation" - if we don't clear them, printing state after THIS run
    # would show a confusing mix of the current query with stale fields from whatever query ran last.
    state.clear()

    state["query"] = query
    state["steps"] = []
    state["result"] = None  # will be filled in once the agent reaches a Final Answer - staying None (instead
    # of some unrelated leftover value) is the honest outcome if we run out of steps without finishing

    scratchpad = ""  # this string GROWS with every Thought/Action/Observation, so the LLM can "remember"
    # what it already tried during this task, even though the LLM itself has no memory between API calls.

    for step_number in range(max_steps):  # the Agent Loop itself - keep taking one step at a time

        messages = [
            {"role": "system", "content": react_prompt},  # the system message sets the rules/behaviour
            {"role": "user", "content": f"User Query: {query}\n\n{scratchpad}\n\nWhat is your next step?"}
            # the user message carries the ACTUAL task, plus everything that has happened so far
            # (the scratchpad). On step 1, scratchpad is empty, so this is just the bare query.
        ]

        llm_output = call_llm(messages)  # ask the LLM to reason about the next step

        try:
            step_data = json.loads(llm_output)  # parse the JSON the model returned, same pattern as before
        except Exception:
            print("Could not parse LLM output as JSON:", llm_output)
            break  # if the model didn't follow the format, we stop rather than crash the whole notebook

        reason = step_data["reason"]
        tool = step_data["tool"]
        tool_input = step_data["input"]

        print(f"\n--- Step {step_number + 1} ---")
        print("Reason:", reason)
        print("Tool:", tool)
        print("Input:", tool_input)

        if tool == "Final Answer":
            # the model has decided it has enough information - this ends the loop early, it does NOT
            # have to run all max_steps every time.
            state["result"] = tool_input
            state["steps"].append({
                "reason": reason, "tool": tool, "input": tool_input, "observation": "N/A"
            })
            print("\nFinal Answer:", tool_input)
            return tool_input

        elif tool == "Calculator":
            calc = CalculatorTool()
            observation = calc.run(tool_input)

        elif tool == "Search":
            search = SearchTool()
            observation = search.run(tool_input)

        else:
            observation = "Unknown tool requested."  # defensive fallback - protects us if the model ever
            # returns a tool name we didn't account for

        print("Observation:", observation)

        # record this step in our state dictionary, same shape as our earlier run_agent() steps
        state["steps"].append({
            "reason": reason, "tool": tool, "input": tool_input, "observation": observation
        })

        # grow the scratchpad with what just happened, so the NEXT loop iteration's LLM call can see it
        scratchpad += f"\nThought: {reason}\nAction: {tool}\nAction Input: {tool_input}\nObservation: {observation}\n"

    # if we exit the for-loop normally (i.e. we never hit "return" above), it means we ran out of steps
    print("\nAgent stopped: reached max_steps without a Final Answer.")
    return None

# What Just Happened?

We wrote `run_react_agent()` — our first *real* agent loop. Compare it to the old `run_agent()`:

| | `run_agent()` (earlier) | `run_react_agent()` (now) |
|---|---|---|
| Tool calls per query | exactly 1 | 1 to `max_steps` |
| Knows about previous steps? | no | yes, via the `scratchpad` |
| Decides when to stop? | always stops after 1 tool | decides itself, via "Final Answer" |
| Can chain Search → Calculator? | no | yes |

The key trick is the `scratchpad` string. Every time we loop, we hand the model a slightly longer "transcript" of Thought/Action/Observation, so it can build on what it already discovered — this is the entire mechanism behind multi-step reasoning.

# Testing the Agent Loop

Let's test it with three queries, from simplest to hardest:

1. A pure calculation (should finish in 1 step)
2. A pure factual lookup (should finish in 1 step)
3. The hard multi-tool case we couldn't solve before: **"Find Chennai's population and calculate its square root"** (should take 2 steps - Search, then Calculator)

In [ ]:
# Test 1: simple calculation - we expect the agent to go straight to "Final Answer" after one Calculator step
run_react_agent("What is 250 * 80?")

In [ ]:
# Test 2: simple factual lookup - we expect one Search step, then "Final Answer"
run_react_agent("Who is the CEO of Google?")

In [ ]:
# Test 3: THE multi-step test - this is the exact task our old hardcoded population_sqrt_workflow() could only
# solve because WE wrote the steps in Python ourselves. Now let's see the agent figure out the same two steps
# (Search, then Calculator) completely on its own, just from reasoning.
run_react_agent("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

If everything worked, the printed output for Test 3 should show something like:

Step 1 → Reason: need the population first → Tool: Search → Observation: (some population number)
   ↓
Step 2 → Reason: now calculate the square root → Tool: Calculator → Observation: (the square root)
   ↓
Final Answer: (the square root)

This is the moment our agent stopped being a single-tool script and became a genuine multi-step reasoner — it chained Search and Calculator together **without us hardcoding that sequence anywhere**. Compare this to our old `population_sqrt_workflow()` function: that one only ever worked for "population + square root" because we wrote those exact two lines of Python. `run_react_agent()` would work just as well for a totally different two-step (or three-step, or four-step) question we never anticipated, as long as Search and Calculator are enough to answer it.

Let's also look at the full `state` dictionary, to see the entire trace of what the agent did:

In [ ]:
print(json.dumps(state, indent=4))
# this prints the full record of the LAST run_react_agent() call - every reason, tool, input, and
# observation, in order. This is incredibly useful for debugging: if the agent ever gives a wrong final
# answer, this trace tells us exactly which step went wrong and why.

# Real-World Problem: Agents Getting Stuck in Search Loops

If you ran Test 2 and Test 3 yourself, you likely saw the agent burn through every step and print "Agent stopped: reached max_steps without a Final Answer" - without ever actually answering, even though the correct answer ("Sundar Pichai", or Chennai's population) was sitting right there in the very FIRST Observation it received.

# Why Did This Happen?

Look closely at what the Search tool actually returns: one long, messy paragraph mashing together snippets from several different web pages, often with conflicting or outdated numbers - "Chennai city" population vs. "Chennai district" population vs. "Chennai metro area" population, or for the CEO example, a mix of historical CEOs (Larry Page, Eric Schmidt) tangled up with the current one (Sundar Pichai), because search snippets don't come neatly labeled "this part is current, this part is old."

Our `react_prompt` never told the model what to do with messy, slightly conflicting information like this. So the model did something very human: it kept searching, hoping the NEXT search would magically return one single, perfectly clean number or name - and it never did, because real-world data is *always* a little messy.

# Why This Matters for Real AI Agents

This is one of the most common failure modes in real production agents - and it's not a bug in the code, it's a **prompting gap**. Tools like web search, databases, or external APIs almost always return noisy, redundant, or slightly conflicting data. A well-designed agent prompt has to explicitly tell the model: "this is normal, here's how to handle it, and here's when to stop searching and commit to an answer." Without that instruction, agents will happily burn through their entire step budget (and your API quota) chasing a "perfect" answer that doesn't exist.

# Our Two-Part Fix

1. **Teach the model how to handle messy data.** We'll add explicit rules to `react_prompt`: if a fact is repeated (even worded differently) across more than one Observation, treat it as confirmed; don't search for the same thing more than twice; prefer a reasonable approximate answer over no answer at all.
2. **Give the model awareness of its own step budget.** Right now the model has no idea it's about to run out of steps. We'll add a short note to every message telling it which step it's on, out of the maximum, so it feels the same "running out of time" pressure we would, and wraps up accordingly.

In [ ]:
# We're redefining react_prompt here, AFTER seeing it fail in testing. This is completely normal in agent
# development - you write a first version, watch how the model actually behaves with real (messy) tool
# outputs, then come back and patch the prompt with rules that address what you saw go wrong. This new
# version OVERWRITES the react_prompt variable from earlier - any cell that runs after this one uses the
# improved version automatically, since run_react_agent() just reads whatever react_prompt currently holds.

react_prompt = """
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

1. Calculator
- Use for mathematical calculations.

2. Search
- Use for factual questions and information retrieval.

3. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{
    "reason": "why you are taking this step",
    "tool": "Calculator" or "Search",
    "input": "the input to give the tool"
}

If you already know the final answer:
{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a tool call you already made.
- Real web search results are messy by nature: they mix old and current information, repeat the same fact in
  different words, and often report slightly different numbers from different sources (for example, a city's
  population vs. its district's population vs. its metro area's population). This is completely normal - do
  NOT keep searching just to find one "perfect" or "exact" number.
- If a name or fact appears in more than one Observation (even worded differently), treat it as confirmed.
  Searching for the exact same thing more than twice is wasteful - after two related searches, commit to your
  best answer instead of searching again.
- An approximate but timely answer is better than no answer. Give the Final Answer as soon as you reasonably
  can.
- You have a limited number of steps, and you will be told which step you're on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you've already gathered, even if it
  isn't perfect.
"""

In [ ]:
# Updating run_react_agent() to fix the looping problem we saw in testing. Two changes from the version
# above:
#   1) default max_steps raised slightly, from 5 to 6, just to give a little extra breathing room
#   2) the user message now tells the model which step it's on, out of the maximum, so it can feel the same
#      "running out of time" pressure we would, and wrap up with a Final Answer accordingly

def run_react_agent(query, max_steps=6):

    state.clear()  # same fix as before - wipe out any leftover keys from a previous run

    state["query"] = query
    state["steps"] = []
    state["result"] = None

    scratchpad = ""

    for step_number in range(max_steps):

        messages = [
            {"role": "system", "content": react_prompt},
            {"role": "user", "content": (
                f"User Query: {query}\n\n{scratchpad}\n\n"
                f"(You are on step {step_number + 1} of {max_steps} maximum steps.)\n\n"
                "What is your next step?"
            )}
            # we added the "(You are on step X of Y maximum steps.)" line so the model always knows how much
            # budget it has left - without this, it has no way of knowing it's about to be cut off.
        ]

        llm_output = call_llm(messages)

        try:
            step_data = json.loads(llm_output)
        except Exception:
            print("Could not parse LLM output as JSON:", llm_output)
            break

        reason = step_data["reason"]
        tool = step_data["tool"]
        tool_input = step_data["input"]

        print(f"\n--- Step {step_number + 1} ---")
        print("Reason:", reason)
        print("Tool:", tool)
        print("Input:", tool_input)

        if tool == "Final Answer":
            state["result"] = tool_input
            state["steps"].append({
                "reason": reason, "tool": tool, "input": tool_input, "observation": "N/A"
            })
            print("\nFinal Answer:", tool_input)
            return tool_input

        elif tool == "Calculator":
            calc = CalculatorTool()
            observation = calc.run(tool_input)

        elif tool == "Search":
            search = SearchTool()
            observation = search.run(tool_input)

        else:
            observation = "Unknown tool requested."

        print("Observation:", observation)

        state["steps"].append({
            "reason": reason, "tool": tool, "input": tool_input, "observation": observation
        })

        scratchpad += f"\nThought: {reason}\nAction: {tool}\nAction Input: {tool_input}\nObservation: {observation}\n"

    print("\nAgent stopped: reached max_steps without a Final Answer.")
    return None

# What Changed

- `react_prompt` now explicitly tells the model that messy/conflicting search results are normal, that repeating a fact across Observations counts as confirmation, and that it should stop after two related searches and commit to an answer.
- `run_react_agent()` now tells the model its exact position in the step budget on every turn (`"You are on step 3 of 6 maximum steps."`), and `max_steps` was bumped from 5 to 6 for a little extra breathing room.

Let's re-run the exact two queries that got stuck before, and see whether the agent now reaches a Final Answer.

In [ ]:
# Retest: this is the exact query that looped through 5 Search calls without answering, last time
run_react_agent("Who is the CEO of Google?")

In [ ]:
# Retest: this is the exact multi-step query that never got past Search to use the Calculator, last time
run_react_agent("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

With the updated prompt and step-budget awareness, both queries should now reach a Final Answer well within the step limit - the CEO question in 1-2 steps, and the Chennai question in 2-3 steps (Search for the population, then Calculator for the square root).

If the agent STILL loops on a particular query, that's a useful signal too: it usually means the underlying search results for that specific query are unusually messy or contradictory, and you could either rephrase the query, increase `max_steps` a bit further, or make the prompt rules even more explicit for that case. This kind of "test → diagnose → patch the prompt → retest" cycle is exactly how real agents get tuned in practice - it's rarely a one-shot process.

# Where We Are Now: A Complete AI Agent

Let's map what we just built back onto the full list of concepts from the start of this notebook:

- **LLM Basics** → done with Groq (`llama-3.1-8b-instant`)
- **Tools** → `CalculatorTool`, `SearchTool`
- **Tool Selection** → the JSON-based prompt that picks a tool
- **State** → the `state` dictionary tracking the whole task
- **ReAct Pattern** → `react_prompt`, teaching the model Thought → Action → Observation
- **Workflow** & **Multi-Step Workflow** → no longer hardcoded - `run_react_agent()` chains tools dynamically
- **Agent Loop** → the `for step_number in range(max_steps)` loop inside `run_react_agent()`
- **Final Answer Generation** → the `if tool == "Final Answer"` branch
- **Tool Calling** → the `elif tool == "Calculator"` / `elif tool == "Search"` branches that actually execute Python code based on the LLM's decision

`run_react_agent()` IS our complete AI agent. Everything from here is about making this same logic cleaner, safer, and better organized — not adding new "intelligence."

# What's Left: Production-Style Refactor

Right now, all our logic lives loosely inside one notebook, calling global variables like `state` and `client` directly. In a real project, you'd want this organized into proper, reusable pieces — e.g. a `Tool` base class, an `Agent` class that owns its own state instead of using a global dictionary, and a clear separation between "prompts," "tools," and "the agent loop." That's our final module: **Refactoring into a Production-Style Structure**.

Let me know when you'd like to continue with that, and we'll wrap up the project.

# Module 11: Refactoring into a Production-Style Structure

# Why Refactor at All?

Everything we built works - but it's built the way you'd build a prototype: loose functions (`run_agent`, `run_react_agent`, `call_llm`) reading and writing a single GLOBAL `state` dictionary and a single GLOBAL `client`. We even ran into a real bug earlier because of this - `state` carried leftover keys between unrelated runs, because every function shared the exact same dictionary.

In a real project, you'd want:
- **No shared global state** - so two different tasks (or two different users, or two different agents) can never accidentally interfere with each other.
- **A consistent interface for tools** - so the agent's loop never needs to know "is this Calculator or Search?" - it just calls `tool.run(input)` and trusts it works, no matter which tool it is.
- **A prompt that builds itself from whatever tools you actually have** - right now, if we added a third tool, we'd have to remember to manually retype its name and description into `react_prompt`. That doesn't scale.

# The Concepts We'll Use: Inheritance & Encapsulation

- **Inheritance** - we'll create one base `Tool` class that defines the "shape" every tool must have (a `name`, a `description`, and a `run()` method). `CalculatorTool` and `SearchTool` will both INHERIT from `Tool`, meaning they automatically get that shape, and just fill in their own specific logic. This is also called **polymorphism**: many different tool classes, one shared interface, so the code that USES the tools doesn't care which one it's talking to.
- **Encapsulation** - we'll wrap the LLM connection, the tools, the prompt, the state, and the loop into ONE `Agent` class. Each `Agent` object owns its own data (`self.state`, `self.client`, etc.) instead of reading and writing variables that float around the whole notebook. This is exactly what fixes our earlier state-leakage bug, permanently - two different `Agent` objects literally cannot share each other's `state`, because each one has its own copy.

# Where This Shows Up in Real Agent Frameworks

This is not just a Python exercise - this is genuinely how real frameworks are structured. LangChain has a `BaseTool` class that every tool inherits from. OpenAI's function-calling API expects every tool to be described by a name + a description + a schema, and an "Agent" or "Assistant" object holds onto its own conversation state. We're rebuilding the same core idea, just at a much smaller, more understandable scale.

In [ ]:
class Tool:
    # this is our base class - think of it as a "contract" that every tool must follow.
    # ANY tool we create (Calculator, Search, or anything we add later - a weather tool, a database
    # lookup tool, etc.) will inherit from this class and is guaranteed to have a .name, a .description,
    # and a .run() method. This means the Agent class never needs to know what a tool actually DOES
    # internally - it just calls tool.run(input) and trusts it'll work, no matter which tool it is.

    def __init__(self, name, description):
        self.name = name  # the tool's name, exactly as it should appear in the prompt (e.g. "Calculator")
        self.description = description  # a short explanation of what the tool does and when to use it -
        # this is what lets us build the "Available Tools" section of our prompt AUTOMATICALLY, instead of
        # hand-typing tool descriptions into the prompt text every single time we add a new tool.

    def run(self, tool_input):
        # this is a placeholder. Every subclass (CalculatorTool, SearchTool, etc.) MUST override this
        # method with its own real logic. If a subclass forgets to override it, calling .run() raises
        # this error instead of silently doing nothing - it's a safety net.
        raise NotImplementedError("Each tool must implement its own run() method.")

In [ ]:
# Redefining CalculatorTool and SearchTool to inherit from Tool. The actual run() logic is UNCHANGED from
# Module 2 - we're only adding the shared name/description "shape" on top of what already worked.

import math  # already imported back in Module 2, re-imported here just so this cell can run on its own

class CalculatorTool(Tool):  # inherits from Tool, so it automatically gets .name and .description for free

    def __init__(self):
        super().__init__(  # super() calls the PARENT class's __init__ (Tool's __init__), so we don't have
            # to repeat the self.name = ... / self.description = ... lines ourselves.
            name="Calculator",
            description="Use for mathematical calculations, e.g. 'sqrt(81)' or '250 * 80'."
        )

    def run(self, expression):  # identical logic to our original CalculatorTool

        try:
            return eval(expression, {"math": math})
        except Exception:
            return "Calculation Error"


class SearchTool(Tool):

    def __init__(self):
        super().__init__(
            name="Search",
            description="Use for factual questions and information retrieval from the web."
        )
        self.search = DuckDuckGoSearchRun()  # identical to our original SearchTool's __init__

    def run(self, query):  # identical logic to our original SearchTool
        return self.search.run(query)

# What Just Happened?

We rewrote `CalculatorTool` and `SearchTool` to inherit from a shared `Tool` base class:

- `super().__init__(name=..., description=...)` hands the name and description up to `Tool`'s constructor, so we don't repeat that boilerplate in every single tool.
- The actual `run()` method in each class is exactly the same logic we already had - we didn't touch the parts that worked.
- Both classes now carry a human-readable `description`. We didn't use it yet, but it's the key piece that lets the next class (`Agent`) build its own prompt automatically.

# Building the Agent Class

This class brings together everything from this entire notebook - the LLM connection, the tools, the ReAct prompt, the state, and the agent loop - into ONE self-contained object.

A few design choices worth calling out before you read the code:

- `self.tools` is a **dictionary** keyed by tool name (e.g. `{"Calculator": <object>, "Search": <object>}`), not a list. This lets `run()` look up the right tool with `self.tools.get(tool_name)` instead of writing a long `if/elif` chain for every tool we might ever add.
- `_build_system_prompt()` starts with an underscore - that's a Python convention meaning "internal helper, not meant to be called from outside the class." It loops over whatever tools were actually passed in and writes their names/descriptions into the prompt for us. Add a third tool later, and its description shows up automatically - no manual prompt editing required.
- `self.state` belongs to `self` (the specific Agent instance), not to the notebook as a whole. Two different `Agent` objects will have two completely separate `state` dictionaries, even if you create them in the same notebook session.

In [ ]:
class Agent:

    def __init__(self, tools, model="llama-3.1-8b-instant", max_steps=6):
        # tools: a list of Tool objects this agent is allowed to use, e.g. [CalculatorTool(), SearchTool()]
        # model: which Groq-hosted model to use - defaults to the one our project requires
        # max_steps: the safety limit / circuit breaker for the agent loop, same idea as run_react_agent()

        groq_api_key = os.getenv("GROQ_API_KEY")
        self.client = Groq(api_key=groq_api_key)  # each Agent creates its own Groq connection

        self.model = model
        self.max_steps = max_steps

        self.tools = {tool.name: tool for tool in tools}  # store tools in a dict keyed by name, so we can
        # look up the right one instantly once the LLM tells us which it picked

        self.state = {}  # each Agent has its OWN state dictionary - this is what fixes the global-state
        # leakage bug we ran into earlier, permanently

        self.system_prompt = self._build_system_prompt()  # build the ReAct instructions ONCE, when the
        # Agent is created, using whatever tools were actually passed in

    def _build_system_prompt(self):

        tools_description = ""  # this becomes the "Available Tools" section of our prompt

        for i, tool in enumerate(self.tools.values(), start=1):
            tools_description += f"{i}. {tool.name}\n- {tool.description}\n\n"
            # for each tool we were given, write its name and description into the prompt automatically -
            # the upgrade over our earlier react_prompt, which had Calculator/Search hardcoded as plain text

        return f"""
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

{tools_description}{len(self.tools) + 1}. Final Answer
- Use this ONLY when you already have enough information to fully answer the user\'s question.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{{
    "reason": "why you are taking this step",
    "tool": "the exact tool name",
    "input": "the input to give the tool"
}}

If you already know the final answer:
{{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don\'t repeat a call you already made.
- Real tool results are messy by nature: they mix old and current information, repeat the same fact in
  different words, and often report slightly different numbers from different sources. This is completely
  normal - do NOT keep using the same tool just to find one "perfect" or "exact" answer.
- If a name or fact appears more than once (even worded differently), treat it as confirmed. Repeating the
  same kind of action more than twice is wasteful - after two related attempts, commit to your best answer.
- An approximate but timely answer is better than no answer.
- You have a limited number of steps, and you will be told which step you\'re on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you\'ve already gathered.
"""

    def _call_llm(self, messages):  # same helper we wrote earlier as a standalone function - now a method,
        # so it automatically uses THIS agent's own client and model instead of relying on globals

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages
        )

        return response.choices[0].message.content

    def run(self, query):
        # this is the same Agent Loop logic from run_react_agent() - but everything now refers to self.*
        # instead of global variables, which is what makes this Agent fully self-contained.

        self.state.clear()  # clean slate every run, same fix as before - now scoped to THIS agent only

        self.state["query"] = query
        self.state["steps"] = []
        self.state["result"] = None

        scratchpad = ""

        for step_number in range(self.max_steps):

            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": (
                    f"User Query: {query}\n\n{scratchpad}\n\n"
                    f"(You are on step {step_number + 1} of {self.max_steps} maximum steps.)\n\n"
                    "What is your next step?"
                )}
            ]

            llm_output = self._call_llm(messages)

            try:
                step_data = json.loads(llm_output)
            except Exception:
                print("Could not parse LLM output as JSON:", llm_output)
                break

            reason = step_data["reason"]
            tool_name = step_data["tool"]
            tool_input = step_data["input"]

            print(f"\n--- Step {step_number + 1} ---")
            print("Reason:", reason)
            print("Tool:", tool_name)
            print("Input:", tool_input)

            if tool_name == "Final Answer":
                self.state["result"] = tool_input
                self.state["steps"].append({
                    "reason": reason, "tool": tool_name, "input": tool_input, "observation": "N/A"
                })
                print("\nFinal Answer:", tool_input)
                return tool_input

            tool = self.tools.get(tool_name)  # look up the actual Tool object by name - this replaces the
            # long if/elif chain we used to need for every single tool

            if tool is None:
                observation = "Unknown tool requested."
            else:
                observation = tool.run(tool_input)  # polymorphism in action - we don't care whether this
                # is CalculatorTool or SearchTool, .run() works the same way for both

            print("Observation:", observation)

            self.state["steps"].append({
                "reason": reason, "tool": tool_name, "input": tool_input, "observation": observation
            })

            scratchpad += f"\nThought: {reason}\nAction: {tool_name}\nAction Input: {tool_input}\nObservation: {observation}\n"

        print("\nAgent stopped: reached max_steps without a Final Answer.")
        return None

# What Just Happened?

We wrote `Agent`, a single class that replaces `client`, `state`, `react_prompt`, `call_llm()`, and `run_react_agent()` all at once:

| Before (loose globals) | After (`Agent` class) |
|---|---|
| `client` (global) | `self.client` |
| `state` (global dict) | `self.state` (one per Agent) |
| `react_prompt` (hand-written text) | `self.system_prompt` (built from `self.tools` automatically) |
| `call_llm(messages)` | `self._call_llm(messages)` |
| `run_react_agent(query)` | `self.run(query)` |
| `if/elif` chain per tool | `self.tools.get(tool_name)` dictionary lookup |

Nothing about the AGENT LOGIC changed - it's the exact same Reason → Act → Observe loop, with the exact same prompt rules we tuned earlier. What changed is purely the *organization*: everything an agent needs now travels together inside one object.

# Testing Our Refactored Agent

Let's create an Agent equipped with both tools, peek at the prompt it auto-generated for itself, then run the same test queries we used before - including the two that gave us trouble earlier - to confirm the refactor didn't break anything.

In [ ]:
agent = Agent(tools=[CalculatorTool(), SearchTool()])  # create one Agent, equipped with both tools

print(agent.system_prompt)  # let's peek at the prompt that got auto-generated from our tools' descriptions

In [ ]:
agent.run("What is 250 * 80?")

In [ ]:
agent.run("Who is the CEO of Google?")

In [ ]:
agent.run("Find Chennai's population and calculate the square root of it.")

In [ ]:
print(json.dumps(agent.state, indent=4))  # the full trace, now living on the Agent object itself

# Why the Refactored Agent Regressed (and the Fix)

If your Chennai test above ran out of steps without a Final Answer, even though the same query worked fine with `run_react_agent()` earlier, here's why - the agent LOGIC didn't change, but two small details did.

# Cause 1: The Prompt Rules Got Too Generic

When the rules moved inside `_build_system_prompt()`, they had to be rewritten to make sense for ANY tool the `Agent` class might ever be given - not just Calculator and Search specifically. In the process, the one CONCRETE example that was doing a lot of the work earlier ("a city's population vs. its district's population vs. its metro area's population") got replaced with a vaguer, more abstract sentence. That sounds like a minor wording change, but smaller/faster models like `llama-3.1-8b-instant` are much better at pattern-matching "oh, this looks like that example" than at reasoning from an abstract rule. We'll restore a concrete example - just phrased generally enough that it still makes sense no matter which tools the Agent is given.

# Cause 2: We Never Set a `temperature`

Every LLM call we've made so far - in every version of this notebook - has left out a parameter called `temperature`. This controls how "random" vs. how "consistent" the model's output is:

- `temperature = 0` → the model always picks the single most likely next token. Same input, (almost) same output, every time.
- higher values (like `0.7` or `1.0`) → the model takes more varied, less predictable paths - great for creative writing, bad for an agent that needs to make consistent, structured decisions.

Without setting it, Groq uses its own default, which leans toward more variation - so even the EXACT same prompt can occasionally take a different reasoning path between runs. For an agent that's choosing tools and deciding when to stop, we want it to be as consistent and predictable as possible, so we'll explicitly set `temperature=0`.

# Applying Both Fixes

We're redefining the `Agent` class below - same overall pattern we've used throughout this notebook (test, find a problem, patch it in a later cell). Because classes work the same way functions do in a notebook, this redefinition only affects `Agent` objects created AFTER this cell runs - so we'll need to recreate `agent` afterward to actually pick up the fix.

In [ ]:
class Agent:

    def __init__(self, tools, model="llama-3.1-8b-instant", max_steps=10):
        # tools: a list of Tool objects this agent is allowed to use, e.g. [CalculatorTool(), SearchTool()]
        # model: which Groq-hosted model to use - defaults to the one our project requires
        # max_steps: the safety limit / circuit breaker for the agent loop. Bumped from 6 to 10 after
        # testing showed that even with a clearer prompt and temperature=0, a genuinely messy query
        # (like the Chennai population one) can still need a few extra exploratory Search steps before
        # the model is confident enough to settle on an answer - consistency and step budget are two
        # separate levers, and both matter.

        groq_api_key = os.getenv("GROQ_API_KEY")
        self.client = Groq(api_key=groq_api_key)  # each Agent creates its own Groq connection

        self.model = model
        self.max_steps = max_steps

        self.tools = {tool.name: tool for tool in tools}  # store tools in a dict keyed by name, so we can
        # look up the right one instantly once the LLM tells us which it picked

        self.state = {}  # each Agent has its OWN state dictionary - this is what fixes the global-state
        # leakage bug we ran into earlier, permanently

        self.system_prompt = self._build_system_prompt()  # build the ReAct instructions ONCE, when the
        # Agent is created, using whatever tools were actually passed in

    def _build_system_prompt(self):

        tools_description = ""  # this becomes the "Available Tools" section of our prompt

        for i, tool in enumerate(self.tools.values(), start=1):
            tools_description += f"{i}. {tool.name}\n- {tool.description}\n\n"
            # for each tool we were given, write its name and description into the prompt automatically -
            # the upgrade over our earlier react_prompt, which had Calculator/Search hardcoded as plain text

        return f"""
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

{tools_description}{len(self.tools) + 1}. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{{
    "reason": "why you are taking this step",
    "tool": "the exact tool name",
    "input": "the input to give the tool"
}}

If you already know the final answer:
{{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a call you already made.
- Real-world tool results are messy by nature: they mix old and current information, repeat the same fact
  in different words, and often report slightly different numbers from different sources - for example, a
  city's population vs. its district's population vs. its metro area's population, or a figure reported in
  different years or currencies. This is completely normal - do NOT keep using the same tool just to find
  one "perfect" or "exact" answer.
- If a name or fact appears more than once (even worded differently), treat it as confirmed. Repeating the
  same kind of action more than twice is wasteful - after two related attempts, commit to your best answer
  instead of trying again.
- An approximate but timely answer is better than no answer. Give the Final Answer as soon as you reasonably
  can.
- You have a limited number of steps, and you will be told which step you're on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you've already gathered, even if it
  isn't perfect.
"""

    def _call_llm(self, messages):  # same helper as before - now also passing a temperature setting

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0  # 0 = always pick the most likely next token, no creative randomness. This makes
            # the agent's tool-selection decisions as consistent and repeatable as possible, run to run -
            # exactly what we want for structured reasoning, as opposed to creative writing tasks.
        )

        return response.choices[0].message.content

    def run(self, query):
        # this is the same Agent Loop logic from run_react_agent() - but everything now refers to self.*
        # instead of global variables, which is what makes this Agent fully self-contained.

        self.state.clear()  # clean slate every run, same fix as before - now scoped to THIS agent only

        self.state["query"] = query
        self.state["steps"] = []
        self.state["result"] = None

        scratchpad = ""

        for step_number in range(self.max_steps):

            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": (
                    f"User Query: {query}\n\n{scratchpad}\n\n"
                    f"(You are on step {step_number + 1} of {self.max_steps} maximum steps.)\n\n"
                    "What is your next step?"
                )}
            ]

            llm_output = self._call_llm(messages)

            try:
                step_data = json.loads(llm_output)
            except Exception:
                print("Could not parse LLM output as JSON:", llm_output)
                break

            reason = step_data["reason"]
            tool_name = step_data["tool"]
            tool_input = step_data["input"]

            print(f"\n--- Step {step_number + 1} ---")
            print("Reason:", reason)
            print("Tool:", tool_name)
            print("Input:", tool_input)

            if tool_name == "Final Answer":
                self.state["result"] = tool_input
                self.state["steps"].append({
                    "reason": reason, "tool": tool_name, "input": tool_input, "observation": "N/A"
                })
                print("\nFinal Answer:", tool_input)
                return tool_input

            tool = self.tools.get(tool_name)  # look up the actual Tool object by name - this replaces the
            # long if/elif chain we used to need for every single tool

            if tool is None:
                observation = "Unknown tool requested."
            else:
                observation = tool.run(tool_input)  # polymorphism in action - we don't care whether this
                # is CalculatorTool or SearchTool, .run() works the same way for both

            print("Observation:", observation)

            self.state["steps"].append({
                "reason": reason, "tool": tool_name, "input": tool_input, "observation": observation
            })

            scratchpad += f"\nThought: {reason}\nAction: {tool_name}\nAction Input: {tool_input}\nObservation: {observation}\n"

        print("\nAgent stopped: reached max_steps without a Final Answer.")
        return None

# What Changed

- The Rules section of the prompt now includes a concrete example again ("a city's population vs. its district's population vs. its metro area's population"), worded generally enough to still apply no matter what tools the Agent has.
- `_call_llm()` now passes `temperature=0` to every request, so the agent's reasoning should be far more consistent run to run.
- The default `max_steps` is also raised from 6 to 10. Testing showed that even with a clearer prompt and consistent reasoning, a genuinely messy query like the Chennai one can still need a handful of extra Search attempts before the model is confident enough to stop. Consistency (temperature) and budget (max_steps) are two separate, complementary levers - improving one doesn't remove the need for the other.

Since `Agent` was just redefined, the OLD `agent` object from earlier in the notebook is still running on the OLD class definition - redefining a class doesn't retroactively change objects that already exist. So let's create a fresh `agent` and re-run the query that was failing.

In [ ]:
# recreate agent using the NEW Agent class definition (the old `agent` object above still belongs to the
# previous version of the class, since it was created before this fix)
agent = Agent(tools=[CalculatorTool(), SearchTool()])

agent.run("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

With the concrete example restored, `temperature=0` in place, and `max_steps` raised to 10, this should now reliably reach a Final Answer - Search for the population, then Calculator for the square root, even if it takes a few extra Search attempts along the way to settle on a number. If it still occasionally loops on a particular query, that's a sign that specific query's search results are unusually messy, and the rules could be made even more explicit for that case - this kind of "test, diagnose, patch the prompt, retest" cycle is exactly how real agents get tuned over time.

# Why Prompting Alone Isn't Enough

If you tested the Chennai query again and it still looped, you may have noticed something specific: the model gave the EXACT same reason ("need to find the most recent one") and ran the EXACT same search ("Chennai population 2023") three steps in a row. Our `react_prompt` / `system_prompt` already says "don't repeat a tool call you already made" and "after two related attempts, commit to your best answer" - but those are just sentences in plain English. A smaller, fast model like `llama-3.1-8b-instant` doesn't always reliably follow soft, natural-language constraints like that, especially under pressure from genuinely messy, repetitive-looking search results.

# The Real-World Lesson

This is one of the most important lessons in building real agents: **prompting shapes behavior, but it doesn't guarantee it.** You can write the clearest rule in the world, and the model can still ignore it. Production agents handle this with "defense in depth" - the prompt is the first layer, asking nicely, but there's also a second layer written in plain Python that physically *cannot* be ignored, because it's not asking the model to behave, it's enforcing the behavior in code.

# Our Fix: Tracking Tool Usage in Code

We'll add two checks inside `run()`, before we ever call a real tool:

1. **Exact duplicate detection** - if the model asks for the SAME tool with the SAME input it already tried, we don't waste a real network call repeating it. Instead, we hand back a pointed message telling it so, and nudging it to try something different or wrap up.
2. **A hard per-tool usage cap** - we count how many times each tool has actually been used this run. Once a tool hits the cap (we'll use 2, matching what our prompt already says), the agent class itself refuses to run it again - no matter what the model asks for - and forces it toward a different tool or a Final Answer instead.

Neither of these touches the LLM call itself - they're just plain Python `if` statements wrapped around the tool-execution step. That's the point: simple, deterministic code is far more reliable than hoping a probabilistic model "remembers" a rule.

In [ ]:
class Agent:

    def __init__(self, tools, model="llama-3.1-8b-instant", max_steps=10):
        groq_api_key = os.getenv("GROQ_API_KEY")
        self.client = Groq(api_key=groq_api_key)

        self.model = model
        self.max_steps = max_steps

        self.tools = {tool.name: tool for tool in tools}

        self.max_uses_per_tool = 2  # HARD cap - matches what we already tell the model in the prompt, but
        # this version is enforced in code, so it can't be ignored the way a prompt instruction can be

        self.state = {}

        self.system_prompt = self._build_system_prompt()

    def _build_system_prompt(self):

        tools_description = ""

        for i, tool in enumerate(self.tools.values(), start=1):
            tools_description += f"{i}. {tool.name}\n- {tool.description}\n\n"

        return f"""
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

{tools_description}{len(self.tools) + 1}. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{{
    "reason": "why you are taking this step",
    "tool": "the exact tool name",
    "input": "the input to give the tool"
}}

If you already know the final answer:
{{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a call you already made.
- Real-world tool results are messy by nature: they mix old and current information, repeat the same fact
  in different words, and often report slightly different numbers from different sources - for example, a
  city's population vs. its district's population vs. its metro area's population, or a figure reported in
  different years or currencies. This is completely normal - do NOT keep using the same tool just to find
  one "perfect" or "exact" answer.
- If a name or fact appears more than once (even worded differently), treat it as confirmed. Each tool can
  only be used twice in this entire task - after that, you must switch tools or give a Final Answer.
- An approximate but timely answer is better than no answer. Give the Final Answer as soon as you reasonably
  can.
- You have a limited number of steps, and you will be told which step you're on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you've already gathered, even if it
  isn't perfect.
"""

    def _call_llm(self, messages):

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0
        )

        return response.choices[0].message.content

    def run(self, query):

        self.state.clear()
        self.state["query"] = query
        self.state["steps"] = []
        self.state["result"] = None

        scratchpad = ""
        tool_usage_count = {}   # e.g. {"Search": 2} - how many times each tool has ACTUALLY been run so far
        seen_calls = set()      # e.g. {("Search", "chennai population")} - exact (tool, input) pairs already tried

        for step_number in range(self.max_steps):

            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": (
                    f"User Query: {query}\n\n{scratchpad}\n\n"
                    f"(You are on step {step_number + 1} of {self.max_steps} maximum steps.)\n\n"
                    "What is your next step?"
                )}
            ]

            llm_output = self._call_llm(messages)

            try:
                step_data = json.loads(llm_output)
            except Exception:
                print("Could not parse LLM output as JSON:", llm_output)
                break

            reason = step_data["reason"]
            tool_name = step_data["tool"]
            tool_input = step_data["input"]

            print(f"\n--- Step {step_number + 1} ---")
            print("Reason:", reason)
            print("Tool:", tool_name)
            print("Input:", tool_input)

            if tool_name == "Final Answer":
                self.state["result"] = tool_input
                self.state["steps"].append({
                    "reason": reason, "tool": tool_name, "input": tool_input, "observation": "N/A"
                })
                print("\nFinal Answer:", tool_input)
                return tool_input

            # normalize the input a bit (lowercase, trimmed) so trivially different casing/spacing still
            # counts as "the same call" for duplicate-detection purposes
            call_signature = (tool_name, str(tool_input).strip().lower())

            if call_signature in seen_calls:
                # GUARDRAIL 1: the model asked for the exact same tool + input it already tried. Don't waste
                # a real tool call repeating it - just tell it so, directly and unambiguously.
                observation = (
                    f"You already made this exact '{tool_name}' call with this input earlier in this task - "
                    f"repeating it will return the same result. Try a meaningfully different input, switch "
                    f"to a different tool, or give your Final Answer now using what you already know."
                )

            elif tool_usage_count.get(tool_name, 0) >= self.max_uses_per_tool:
                # GUARDRAIL 2: this tool has hit its hard usage cap. We refuse to run it again, no matter
                # what the model asked for - this is enforced here in Python, not left up to the model.
                observation = (
                    f"You have already used '{tool_name}' {tool_usage_count[tool_name]} times in this task, "
                    f"which is the maximum allowed. You may NOT use it again. Use a different tool, or give "
                    f"your Final Answer now with your best estimate from what you've already gathered."
                )

            else:
                tool = self.tools.get(tool_name)

                if tool is None:
                    observation = "Unknown tool requested."
                else:
                    observation = tool.run(tool_input)  # only NOW do we actually call the real tool
                    tool_usage_count[tool_name] = tool_usage_count.get(tool_name, 0) + 1
                    seen_calls.add(call_signature)

            print("Observation:", observation)

            self.state["steps"].append({
                "reason": reason, "tool": tool_name, "input": tool_input, "observation": observation
            })

            scratchpad += f"\nThought: {reason}\nAction: {tool_name}\nAction Input: {tool_input}\nObservation: {observation}\n"

        print("\nAgent stopped: reached max_steps without a Final Answer.")
        return None

# What Changed

- `self.max_uses_per_tool = 2` is a hard limit, checked in plain Python before any tool actually runs.
- `seen_calls` catches exact repeats (same tool, same input) and intercepts them with a pointed message instead of silently repeating the same search.
- `tool_usage_count` enforces the cap even when the model asks for slightly DIFFERENT inputs each time (like "Chennai population 2023" vs. "Chennai population 2023 latest numbers") - after 2 real uses of a tool, the agent refuses the 3rd, regardless of wording.

Note the difference in spirit from before: previously, every fix we made (the concrete example, `temperature=0`) was about persuading the model to behave better. This fix doesn't persuade - it constrains. Even if the model "wants" to search a third time, it physically can't; it gets a forced message back instead. Let's recreate `agent` from this new class and retest.

In [ ]:
agent = Agent(tools=[CalculatorTool(), SearchTool()])  # recreated again, now using the guardrail-enforcing class

agent.run("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

Even if the model still TRIES to repeat a search (which it might - we haven't changed its underlying tendency, only what happens when it tries), you should now see it get redirected after at most 2 real searches, instead of silently repeating the same query 3-4+ times. This is the more reliable kind of fix: it doesn't depend on the model "deciding" to behave - the agent's own code makes the undesired behavior impossible past a certain point.

# A Real Bug: "Calculation Error" Isn't Random

Look closely at a failing Calculator step from testing: `Input: sqrt(4646732)` → `Observation: Calculation Error`. This isn't bad luck - it's a genuine bug in `CalculatorTool.run()`, and it's worth understanding exactly why, since it's a classic mistake that's easy to make with Python's `eval()`.

# The Root Cause

Our `run()` method calls `eval(expression, {"math": math})`. That second argument tells `eval()` exactly which names are allowed to exist while it evaluates the expression - and we only gave it ONE name: `math` (the whole module, as a single object). That means `math.sqrt(81)` works fine, because `math` exists in that namespace and `.sqrt` is just an attribute of it. But a bare `sqrt(81)` does NOT work, because the standalone name `sqrt` was never added - only `math` was.

Here's the part that makes this genuinely confusing to debug: our tool's own `description` says `"e.g. 'sqrt(81)'"` - so we're actively telling the LLM to write the ONE form of input that will always fail. Whenever the model happens to write `math.sqrt(...)` it works; whenever it follows our own example and writes `sqrt(...)`, it breaks. That's exactly the inconsistency you noticed - "sometimes we can reach answer and sometimes we can't" - it was never about luck, it was about which exact wording the model happened to choose that step.

# Why This Compounds Into a Bigger Problem

Once `Calculator` returns `"Calculation Error"`, two bad things tend to happen next, both visible in your traces:
1. The model often retries the EXACT same broken expression, which then trips our duplicate-call guardrail - wasting a step on a problem the guardrail was never meant to solve.
2. If the model gives up on `Calculator` entirely, it sometimes just computes the square root "in its head" inside the Final Answer text instead of using the tool - and LLMs are notoriously unreliable at mental arithmetic on large numbers. That's why one of your "successful" runs actually gave a WRONG final answer (≈3429, when the real square root of ~4.6 million is ≈2155) - it looked like a win because it reached a Final Answer, but the math itself was never verified by the Calculator tool at all.

# The Fix

We'll rebuild the `eval()` namespace so it includes EVERY function and constant from the `math` module directly by name (`sqrt`, `pow`, `log`, `pi`, etc.) - not just the `math` object itself. That way, both `sqrt(81)` and `math.sqrt(81)` work correctly, matching what our own tool description already promises the model.

In [ ]:
# quick reproduction first, so we can SEE the bug before fixing it - this is good debugging practice:
# confirm you understand exactly why something fails before you patch it.

import math

try:
    eval("sqrt(4646732)", {"math": math})
except Exception as e:
    print("sqrt(4646732) with only 'math' in scope fails:", type(e).__name__, "-", e)
    # NameError: name 'sqrt' is not defined - 'math' the module is in scope, but the bare name
    # 'sqrt' was never added to the namespace, so Python has no idea what 'sqrt' refers to.

In [ ]:
class CalculatorTool(Tool):

    def __init__(self):
        super().__init__(
            name="Calculator",
            description="Use for mathematical calculations, e.g. 'sqrt(81)' or '250 * 80'."
        )

        # building a namespace that includes EVERY public function/constant from the math module,
        # directly by name (sqrt, pow, log, sin, cos, pi, e, ...) - not just the 'math' object itself.
        # dir(math) lists every name defined inside the math module; we filter out the "dunder" names
        # (the ones starting with "_", like __name__) since those are Python internals, not real
        # functions we want exposed to eval().
        self.eval_namespace = {name: getattr(math, name) for name in dir(math) if not name.startswith("_")}
        self.eval_namespace["math"] = math  # we ALSO keep "math" itself in scope, so math.sqrt(81) still
        # works too - this fix is purely additive, nothing that worked before stops working.

    def run(self, expression):

        try:
            return eval(expression, self.eval_namespace)  # now both 'sqrt(81)' AND 'math.sqrt(81)' work,
            # because the namespace this time actually matches what our own description promises the model.
        except Exception:
            return "Calculation Error"

In [ ]:
# re-verify the fix directly, the same way we reproduced the bug above
calc = CalculatorTool()

print("sqrt(4646732)      ->", calc.run("sqrt(4646732)"))       # bare form - this used to fail
print("math.sqrt(4646732) ->", calc.run("math.sqrt(4646732)"))  # dotted form - already worked, still works
print("250 * 80            ->", calc.run("250 * 80"))            # plain arithmetic - unaffected by this fix

# What Just Happened?

We fixed `CalculatorTool` by building a namespace dictionary out of every public name in the `math` module (`{"sqrt": math.sqrt, "pow": math.pow, "pi": 3.14159..., ...}`), instead of only exposing the `math` module itself. Now `sqrt(4646732)` works correctly and returns a real numeric answer instead of `"Calculation Error"` - which also means the model no longer needs to retry the same broken call (avoiding the wasted guardrail interception) or fall back to unreliable mental math in its Final Answer.

Since `CalculatorTool` was just redefined, we need to recreate `agent` one more time so it picks up the fixed tool - then retest the Chennai query, this time with a working calculator.

In [ ]:
agent = Agent(tools=[CalculatorTool(), SearchTool()])  # recreated again - now using the FIXED CalculatorTool

agent.run("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

This should now complete cleanly: Search for the population (at most twice, thanks to our guardrail), then a Calculator call that actually succeeds the first time, then a Final Answer with a real, verified square root - not a guess.

This whole detour - prompt fix, then temperature, then max_steps, then a code-level guardrail, then finally this `eval()` namespace bug - is a genuinely realistic picture of what building an agent looks like in practice. Bugs hide at every layer: the prompt, the model's behavior, the loop's safety limits, and the tools' own implementation. A wrong final answer can come from any of them, and it often takes looking at several real traces (exactly like the ones you pasted) before the true root cause becomes clear.

# Two More Real Bugs, Found From a Single Failing Trace

A harder query - "Find Chennai's population and calculate the square root of it" - exposed two more issues, both visible in one trace:

1. **The Calculator received prose, not math.** The model's final attempt was `Input: growth rate of 1.08 (from 46.47 lakh to 50.08 lakh in 2036) applied to 46.47 lakh` - that's an English sentence, not a Python expression. `eval()` can only run actual code; it has no way to interpret a description of a calculation, so this always returns `"Calculation Error"`, no matter how well-meaning the description was. This is different from our earlier `sqrt()` namespace bug - that was a real gap in what the calculator COULD run; this is the model not translating its own reasoning into valid math syntax before sending it.
2. **A wasted step repeating an already-blocked request.** After Search hit its usage cap, the model asked for the EXACT same blocked Search call again on the very next step - spending an entire LLM call to get told "no" a second time, instead of moving on immediately.

# Why This Happens

Some of this is genuinely just query difficulty - Chennai's population doesn't have one clean, agreed-upon number on the open web (it varies by city/district/metro and by year), so the model has to improvise a multi-step plan (search, then estimate using a growth rate) instead of a simple search-then-calculate. The more improvised the plan, the more likely the model is to write something the Calculator can't parse.

But part of it is also something we can genuinely fix: our `Calculator` description never explicitly said "your input must be valid Python code, not a sentence" - and our cap-block message, while accurate, wasn't forceful enough to stop the model from trying the identical blocked request again immediately.

# The Fix

1. Make the Calculator's description and the prompt's Rules explicitly require a valid math expression (numbers, operators, functions only) - with a concrete example of what NOT to send.
2. Make blocked Search/Calculator attempts get remembered the same way successful ones do, and make the block message more directive ("do not attempt this again"), so a repeated blocked request is recognized instantly and discouraged more strongly.

In [ ]:
class CalculatorTool(Tool):

    def __init__(self):
        super().__init__(
            name="Calculator",
            description=(
                "Use for mathematical calculations. Input MUST be a valid Python math expression made only "
                "of numbers, operators (+ - * / ** //), and functions like sqrt(x), pow(x, y), log(x). "
                "Do the reasoning yourself first, then send ONLY the final numeric expression - e.g. send "
                "'4647000 * (5008/4647)', never a sentence describing the calculation."
            )
        )
        self.eval_namespace = {name: getattr(math, name) for name in dir(math) if not name.startswith("_")}
        self.eval_namespace["math"] = math

    def run(self, expression):
        try:
            return eval(expression, self.eval_namespace)
        except Exception:
            return "Calculation Error"

In [ ]:
class Agent:

    def __init__(self, tools, model="llama-3.1-8b-instant", max_steps=10):
        groq_api_key = os.getenv("GROQ_API_KEY")
        self.client = Groq(api_key=groq_api_key)

        self.model = model
        self.max_steps = max_steps

        self.tools = {tool.name: tool for tool in tools}
        self.max_uses_per_tool = 2

        self.state = {}
        self.system_prompt = self._build_system_prompt()

    def _build_system_prompt(self):

        tools_description = ""
        for i, tool in enumerate(self.tools.values(), start=1):
            tools_description += f"{i}. {tool.name}\n- {tool.description}\n\n"

        return f"""
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

{tools_description}{len(self.tools) + 1}. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{{
    "reason": "why you are taking this step",
    "tool": "the exact tool name",
    "input": "the input to give the tool"
}}

If you already know the final answer:
{{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a call you already made,
  and don't repeat a call you were already told you cannot make again.
- For Calculator specifically: input must be a valid Python math expression using ONLY numbers, operators,
  and math functions - never a sentence or description. Work out the numbers in your own reasoning first,
  then send just the final expression, e.g. '4647000 * (5008/4647)', not 'apply the growth rate to the
  population'.
- Real-world tool results are messy by nature: they mix old and current information, repeat the same fact
  in different words, and often report slightly different numbers from different sources - for example, a
  city's population vs. its district's population vs. its metro area's population, or a figure reported in
  different years or currencies. This is completely normal - do NOT keep using the same tool just to find
  one "perfect" or "exact" answer.
- If a name or fact appears more than once (even worded differently), treat it as confirmed. Each tool can
  only be used twice in this entire task - after that, you must switch tools or give a Final Answer.
- An approximate but timely answer is better than no answer. Give the Final Answer as soon as you reasonably
  can.
- You have a limited number of steps, and you will be told which step you're on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you've already gathered, even if it
  isn't perfect.
"""

    def _call_llm(self, messages):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0
        )
        return response.choices[0].message.content

    def run(self, query):

        self.state.clear()
        self.state["query"] = query
        self.state["steps"] = []
        self.state["result"] = None

        scratchpad = ""
        tool_usage_count = {}
        seen_calls = set()  # now tracks BOTH calls that actually ran AND calls that got blocked by the cap -
        # so a repeat of either kind is recognized instantly, instead of re-deriving the same block twice

        for step_number in range(self.max_steps):

            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": (
                    f"User Query: {query}\n\n{scratchpad}\n\n"
                    f"(You are on step {step_number + 1} of {self.max_steps} maximum steps.)\n\n"
                    "What is your next step?"
                )}
            ]

            llm_output = self._call_llm(messages)

            try:
                step_data = json.loads(llm_output)
            except Exception:
                print("Could not parse LLM output as JSON:", llm_output)
                break

            reason = step_data["reason"]
            tool_name = step_data["tool"]
            tool_input = step_data["input"]

            print(f"\n--- Step {step_number + 1} ---")
            print("Reason:", reason)
            print("Tool:", tool_name)
            print("Input:", tool_input)

            if tool_name == "Final Answer":
                self.state["result"] = tool_input
                self.state["steps"].append({
                    "reason": reason, "tool": tool_name, "input": tool_input, "observation": "N/A"
                })
                print("\nFinal Answer:", tool_input)
                return tool_input

            call_signature = (tool_name, str(tool_input).strip().lower())

            if call_signature in seen_calls:
                # this covers BOTH "you already ran this successfully" AND "you already got blocked trying
                # this" - either way, repeating it again is pointless, so we say so immediately.
                observation = (
                    f"You already attempted this exact '{tool_name}' call with this input earlier in this "
                    f"task (whether it ran or was blocked) - repeating it again will not help. Try a "
                    f"meaningfully different input, switch to a different tool, or give your Final Answer now."
                )

            elif tool_usage_count.get(tool_name, 0) >= self.max_uses_per_tool:
                observation = (
                    f"You have already used '{tool_name}' {tool_usage_count[tool_name]} times in this task, "
                    f"which is the maximum allowed. Do NOT attempt to use '{tool_name}' again for the rest "
                    f"of this task. Switch to a different tool right now, or give your Final Answer "
                    f"immediately with your best estimate from what you've already gathered."
                )
                seen_calls.add(call_signature)  # remember this blocked attempt too, so an identical retry
                # is caught by the faster check above instead of re-deriving the same cap message again

            else:
                tool = self.tools.get(tool_name)

                if tool is None:
                    observation = "Unknown tool requested."
                else:
                    observation = tool.run(tool_input)
                    tool_usage_count[tool_name] = tool_usage_count.get(tool_name, 0) + 1
                    seen_calls.add(call_signature)

            print("Observation:", observation)

            self.state["steps"].append({
                "reason": reason, "tool": tool_name, "input": tool_input, "observation": observation
            })

            scratchpad += f"\nThought: {reason}\nAction: {tool_name}\nAction Input: {tool_input}\nObservation: {observation}\n"

        print("\nAgent stopped: reached max_steps without a Final Answer.")
        return None

# What Changed

- `CalculatorTool`'s description now explicitly says input must be numbers/operators/functions only, with a concrete example of what NOT to send (a sentence).
- The Rules section repeats this constraint specifically for Calculator, since tool descriptions and general rules don't always get equal attention from the model - repeating an important constraint in two places makes it more likely to stick.
- `seen_calls` now also records calls that got blocked by the usage cap, not just calls that actually ran. A literal repeat of a blocked request is now caught by the same fast "you already tried this" check, and the cap message itself is more direct about not retrying.

Let's recreate `agent` once more and retest.

In [ ]:
agent = Agent(tools=[CalculatorTool(), SearchTool()])  # recreated again, picking up both fixes

agent.run("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

The Calculator should now reliably receive valid math expressions instead of prose, and repeated blocked requests should stop wasting extra steps. That said, it's worth being honest about something here: this specific query (Chennai's population) will likely always be somewhat unpredictable, simply because the underlying web data itself is messy and inconsistent - that's a property of the real world, not something any amount of code can fully eliminate. What we CAN guarantee through code is that the agent handles that messiness gracefully: it won't crash, it won't loop wastefully, and it will give an honest, clearly-labeled estimate rather than a confident wrong answer or no answer at all.

# A New Bug: Writing the Formula Instead of Solving It

A real run on the Chennai query ended with this Final Answer: `"sqrt(13109989)"`. Look closely - that's not an answer, it's an unevaluated formula. The model decided it had "enough information" and picked Final Answer, but instead of actually running that expression through Calculator first, it just typed the formula out as if writing it down were the same thing as solving it.

This is a genuinely different gap from the ones we already fixed. It's not the model repeating itself, and it's not sending invalid syntax to a tool - the expression `sqrt(13109989)` is perfectly valid math, it's just never been computed. The model treated "I know what calculation to do" as equivalent to "I have done the calculation," which isn't true.

# Why a Prompt Rule Alone Might Not Be Enough

We could add a rule saying "never submit an unevaluated expression as your Final Answer" - and we will, as one layer of the fix. But by now we've seen this pattern several times in this notebook: a soft, natural-language instruction is a good first layer, but it doesn't GUARANTEE compliance from a small, fast model. We want a second layer that can't be talked around.

# The Fix: Verifying the Final Answer With Code We Already Trust

Here's the elegant part - we don't need to build anything new to catch this. We already have a `CalculatorTool` with a safe, working `eval_namespace` (the same one we fixed earlier for the `sqrt()` bug). We can reuse THAT to double-check the Final Answer itself, the moment the model submits it:

- Try to evaluate the Final Answer text the same way `Calculator.run()` would.
- If it succeeds AND the result is a different value than the text itself (meaning it really was a raw, uncomputed expression) - we know the model just wrote a formula. We compute the real number ourselves and attach it to the answer, so the person reading it gets an actual value, not homework left unfinished.
- If it fails (which is the NORMAL case - most final answers are sentences, not bare math expressions) - we simply leave the answer untouched. A failed `eval()` here isn't a problem, it's the expected outcome for a real, already-composed answer.

This is a nice general pattern worth remembering: the same tool you built to DO something can often be reused to VERIFY something, for free.

In [ ]:
class Agent:

    def __init__(self, tools, model="llama-3.1-8b-instant", max_steps=10):
        groq_api_key = os.getenv("GROQ_API_KEY")
        self.client = Groq(api_key=groq_api_key)

        self.model = model
        self.max_steps = max_steps

        self.tools = {tool.name: tool for tool in tools}
        self.max_uses_per_tool = 2

        self.state = {}
        self.system_prompt = self._build_system_prompt()

    def _build_system_prompt(self):

        tools_description = ""
        for i, tool in enumerate(self.tools.values(), start=1):
            tools_description += f"{i}. {tool.name}\n- {tool.description}\n\n"

        return f"""
You are an AI Agent that thinks step-by-step using the ReAct (Reason + Act) pattern.

Available Tools:

{tools_description}{len(self.tools) + 1}. Final Answer
- Use this ONLY when you already have enough information to fully answer the user's question.
- Your Final Answer input must be the actual resolved answer, in plain language - NEVER an unevaluated
  mathematical expression like 'sqrt(13109989)' or '4647000 * 1.08'. If a calculation is still needed,
  use Calculator first to get the real number, THEN give that number as your Final Answer.

You will be shown the conversation so far, including any earlier Thoughts, Actions, and Observations.
At every step, decide what to do next and return ONLY a JSON object (no markdown fences, no extra text)
in ONE of these two formats:

If you still need to use a tool:
{{
    "reason": "why you are taking this step",
    "tool": "the exact tool name",
    "input": "the input to give the tool"
}}

If you already know the final answer:
{{
    "reason": "why you now have enough information",
    "tool": "Final Answer",
    "input": "the final answer to give the user"
}}

Rules:
- Use only ONE tool per step.
- Always check the Observations already shown to you before acting - don't repeat a call you already made,
  and don't repeat a call you were already told you cannot make again.
- For Calculator specifically: input must be a valid Python math expression using ONLY numbers, operators,
  and math functions - never a sentence or description. Work out the numbers in your own reasoning first,
  then send just the final expression, e.g. '4647000 * (5008/4647)', not 'apply the growth rate to the
  population'.
- Real-world tool results are messy by nature: they mix old and current information, repeat the same fact
  in different words, and often report slightly different numbers from different sources - for example, a
  city's population vs. its district's population vs. its metro area's population, or a figure reported in
  different years or currencies. This is completely normal - do NOT keep using the same tool just to find
  one "perfect" or "exact" answer.
- If a name or fact appears more than once (even worded differently), treat it as confirmed. Each tool can
  only be used twice in this entire task - after that, you must switch tools or give a Final Answer.
- An approximate but timely answer is better than no answer. Give the Final Answer as soon as you reasonably
  can.
- You have a limited number of steps, and you will be told which step you're on. As you approach your last
  step, you MUST wrap up with a Final Answer using whatever information you've already gathered, even if it
  isn't perfect.
"""

    def _call_llm(self, messages):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0
        )
        return response.choices[0].message.content

    def run(self, query):

        self.state.clear()
        self.state["query"] = query
        self.state["steps"] = []
        self.state["result"] = None

        scratchpad = ""
        tool_usage_count = {}
        seen_calls = set()

        for step_number in range(self.max_steps):

            messages = [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": (
                    f"User Query: {query}\n\n{scratchpad}\n\n"
                    f"(You are on step {step_number + 1} of {self.max_steps} maximum steps.)\n\n"
                    "What is your next step?"
                )}
            ]

            llm_output = self._call_llm(messages)

            try:
                step_data = json.loads(llm_output)
            except Exception:
                print("Could not parse LLM output as JSON:", llm_output)
                break

            reason = step_data["reason"]
            tool_name = step_data["tool"]
            tool_input = step_data["input"]

            print(f"\n--- Step {step_number + 1} ---")
            print("Reason:", reason)
            print("Tool:", tool_name)
            print("Input:", tool_input)

            if tool_name == "Final Answer":
                final_text = str(tool_input)

                # SAFETY NET: check whether the model just submitted a raw, unevaluated expression
                # (like "sqrt(13109989)") instead of an actual answer. We reuse CalculatorTool's own
                # eval_namespace to try computing it - if that succeeds AND gives a genuinely different
                # value than the text itself, we know it was an uncomputed formula, so we resolve it for
                # real before accepting it. If eval() fails, that's the NORMAL case (a real answer isn't
                # valid Python), so we leave the text untouched.
                calculator = self.tools.get("Calculator")
                if calculator is not None:
                    try:
                        computed_value = eval(final_text, calculator.eval_namespace)
                        if isinstance(computed_value, (int, float)) and str(computed_value) != final_text.strip():
                            final_text = f"{tool_input} = {computed_value}"
                    except Exception:
                        pass  # not a raw expression - this is expected for almost every real answer

                self.state["result"] = final_text
                self.state["steps"].append({
                    "reason": reason, "tool": tool_name, "input": tool_input, "observation": "N/A"
                })
                print("\nFinal Answer:", final_text)
                return final_text

            call_signature = (tool_name, str(tool_input).strip().lower())

            if call_signature in seen_calls:
                observation = (
                    f"You already attempted this exact '{tool_name}' call with this input earlier in this "
                    f"task (whether it ran or was blocked) - repeating it again will not help. Try a "
                    f"meaningfully different input, switch to a different tool, or give your Final Answer now."
                )

            elif tool_usage_count.get(tool_name, 0) >= self.max_uses_per_tool:
                observation = (
                    f"You have already used '{tool_name}' {tool_usage_count[tool_name]} times in this task, "
                    f"which is the maximum allowed. Do NOT attempt to use '{tool_name}' again for the rest "
                    f"of this task. Switch to a different tool right now, or give your Final Answer "
                    f"immediately with your best estimate from what you've already gathered."
                )
                seen_calls.add(call_signature)

            else:
                tool = self.tools.get(tool_name)

                if tool is None:
                    observation = "Unknown tool requested."
                else:
                    observation = tool.run(tool_input)
                    tool_usage_count[tool_name] = tool_usage_count.get(tool_name, 0) + 1
                    seen_calls.add(call_signature)

            print("Observation:", observation)

            self.state["steps"].append({
                "reason": reason, "tool": tool_name, "input": tool_input, "observation": observation
            })

            scratchpad += f"\nThought: {reason}\nAction: {tool_name}\nAction Input: {tool_input}\nObservation: {observation}\n"

        print("\nAgent stopped: reached max_steps without a Final Answer.")
        return None

# What Changed

- Added an explicit Final Answer rule to the prompt: the answer must be resolved, plain language - never a raw expression.
- Added a code-level safety net: right when a Final Answer is accepted, we try to evaluate it using `Calculator`'s own `eval_namespace`. If the text turns out to be a real, uncomputed expression, we compute it and attach the actual number; if it's normal prose (almost always the case), `eval()` fails harmlessly and nothing changes.

This is a nice illustration of "defense in depth" done cheaply: no extra LLM call, no extra step, no risk to normal answers - we're just double-checking the output using a tool we already built and trust.

In [ ]:
agent = Agent(tools=[CalculatorTool(), SearchTool()])  # recreated again, picking up this fix

agent.run("Find Chennai's population and calculate the square root of it.")

# What Just Happened?

If the model again tries to submit a bare expression as its Final Answer (which the prompt rule alone doesn't fully prevent), the code-level check should now catch it and resolve it into a real, computed number before you ever see the result - rather than handing back an unfinished formula.

# Proving the State-Leakage Bug Is Actually Fixed

Earlier in this notebook, we hit a real bug: a single global `state` dict meant two different runs could leave stale data behind for each other. Let's prove that's now structurally impossible - we'll create a SECOND, completely independent Agent (with a different toolset and a tighter step budget) and run it right alongside the first one.

In [ ]:
agent2 = Agent(tools=[CalculatorTool()], max_steps=3)  # a second agent that ONLY has access to a
# calculator, with a tighter step budget - totally separate from `agent` above

agent2.run("What is the square root of 144?")

print("agent's last query: ", agent.state["query"])    # still shows agent's own last query, completely
# unaffected by anything agent2 just did
print("agent2's last query:", agent2.state["query"])   # shows agent2's own query

# What Just Happened?

`agent` and `agent2` ran completely independently, with no risk of one overwriting the other's `state` - because each `Agent` instance carries its own separate dictionary. With our old global-variable design, running two "agents" like this side by side wasn't even possible without them colliding. This is the concrete payoff of encapsulation: it doesn't just look tidier, it removes an entire category of bug.

# Project Complete: A Full AI Agent, Built From Scratch

Here's the entire journey this notebook took, end to end:

1. **LLM Basics** - learned to send a message to an LLM (Groq's `llama-3.1-8b-instant`) and read back a reply.
2. **Tools** - built `CalculatorTool` and `SearchTool`, each wrapping a real capability (math evaluation, live web search) behind a simple `.run()` method.
3. **Tool Selection** - taught the LLM to read a query and choose which tool to use, via a structured JSON prompt.
4. **State** - introduced a dictionary to track what the agent was doing and why, across a single task.
5. **ReAct Pattern** - upgraded the prompt so the model could reason in a loop: Thought → Action → Observation → repeat → Final Answer.
6. **Workflow** & **7. Multi-Step Workflow** - first hardcoded a two-step task ourselves, then watched the agent learn to chain Search → Calculator on its own, with no hardcoding.
8. **Agent Loop** - wrote the `for` loop that repeatedly calls the LLM, runs whatever tool it picks, and feeds the result back in, until the model is ready to answer.
9. **Final Answer Generation** - the branch inside that same loop where the model decides it has enough information and stops.
10. **Complete AI Agent** - `run_react_agent()`, our first fully working multi-step agent - which we then debugged twice: once for a state-leakage bug, once for getting stuck in search loops on messy real-world data.
11. **Production-Style Refactor** - rebuilt the same logic as a `Tool` base class plus an `Agent` class, removing all global state and making the prompt build itself from whatever tools are registered.

# Where You Could Take This Next

A few natural directions, if you want to keep going beyond this notebook:

- **More tools** - a weather tool, a calendar tool, a file-reading tool. Thanks to the `Tool` base class, adding one is just: write a class with a `name`, `description`, and `run()`, then pass it into `Agent(tools=[...])`.
- **Persistent memory** - right now `self.state` resets on every `.run()` call. A real assistant often needs to remember earlier conversations - that usually means saving state to a file or database instead of just an in-memory dictionary.
- **Multi-agent systems** - multiple `Agent` objects, each with different tools, coordinating on a larger task (this is the core idea behind frameworks like CrewAI and LangGraph).
- **Deploying it** - wrapping `Agent` in a small web server (e.g. with FastAPI) so other applications could send it a query over the internet and get a Final Answer back, instead of only running inside a notebook.
- **Learning a framework** - now that you've built the core ReAct loop, tool interface, and state management by hand, frameworks like LangChain or LangGraph will make a lot more sense - they're solving the exact same problems you just solved, just with more configuration options and more built-in tools.

Nice work building a complete AI agent from scratch.